# Privacy Guardian — Colab GPU Worker + Ollama AI

Optional temporary compute runtime for **approved heavy work only**
(Ollama reasoning, vision/OCR parsing). The laptop remains the
**system of record**. Colab is disposable — it never stores the evidence DB.

**What you need before running:**
1. Laptop backend running (`uv run uvicorn app.backend.main:app --host 127.0.0.1 --port 8000`)
2. Laptop's port 8000 exposed on a tunnel (e.g. Cloudflare) so this notebook can poll it
3. That tunnel URL pasted into `COLAB_JOB_DISPATCHER_URL` in the Configuration cell

Optional: expose Colab's Ollama (port 11434) on another tunnel so the laptop's
ModelRouter can call it directly — see the final cell.

## 1 · Configuration

In [ ]:
# ===== CONFIGURATION =====
REPO_URL = "https://github.com/S-Q-Ali/Mis-Clear.git"
# Point at the laptop's API through a tunnel, e.g.
#   https://xxx.trycloudflare.com/api
COLAB_JOB_DISPATCHER_URL = ""
MAX_ITERATIONS = 30        # jobs to process per run
POLL_INTERVAL_S = 5
PROBE_GPU = True
RUN_DIR = "/content/privacy-guardian"
OLLAMA_MODELS = ["qwen3:8b", "gemma3:4b"]

## 2 · Install Ollama (AI Server)

In [ ]:
# === Install Ollama (Colab Linux) ===
import subprocess, os, time, urllib.request

print("Installing Ollama...")
subprocess.run(["curl", "-L", "--fail", "-o", "/usr/local/bin/ollama",
              "https://ollama.com/download/ollama-linux-amd64"], check=True)
subprocess.run(["chmod", "+x", "/usr/local/bin/ollama"], check=True)

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"
_proc = subprocess.Popen(
    ["ollama", "serve"],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
time.sleep(5)

try:
    urllib.request.urlopen("http://127.0.0.1:11434/api/tags", timeout=10)
    print("Ollama server is live!")
except Exception as e:
    print(f"Server starting (may take a moment): {e}")

## 3 · Download AI Models (GPU Accelerated)

In [ ]:
# === Download Models ===
import httpx

for model in OLLAMA_MODELS:
    print(f"Downloading {model}...")
    try:
        with httpx.stream("POST", "http://127.0.0.1:11434/api/pull",
                      json={"name": model}, timeout=600) as resp:
            for line in resp.iter_lines():
                if line:
                    print(f"  {line[:80]}")
        print(f"{model} ready!")
    except Exception as e:
        print(f"Error: {e}")

resp = httpx.get("http://127.0.0.1:11434/api/tags", timeout=10)
print("Available:", [m["name"] for m in resp.json().get("models", [])])

## 4 · Clone repo + install deps

In [ ]:
# === Clone repo + deps ===
import os, subprocess, sys

os.makedirs("/content", exist_ok=True)
if not os.path.isdir(os.path.join(RUN_DIR, ".git")):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, RUN_DIR], check=True)
else:
    subprocess.run(["git", "-C", RUN_DIR, "pull", "--ff-only"], check=True)

sys.path.insert(0, RUN_DIR)

# Worker transport only needs httpx (stdlib + dataclasses for the rest).
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "httpx"], check=True)
print(f"Repo ready at {RUN_DIR}")

## 5 · Capability probe

In [ ]:
# === Advertise what this worker can do ===
from colab.capabilities import detect_capabilities

CAPS = detect_capabilities(probe_gpu=PROBE_GPU)
import json
print(json.dumps(CAPS.advertise(), indent=2))

if CAPS.gpu:
    print("GPU detected \u2014 vision/OCR handlers will run.")
else:
    print("No GPU detected \u2014 vision/OCR handlers will report honest blocked results.")

## 6 · Run dispatcher loop (worker → laptop)

In [ ]:
# === Poll-proc-ack loop ===
if not COLAB_JOB_DISPATCHER_URL:
    print("COLAB_JOB_DISPATCHER_URL is empty.")
    print("Add the laptop tunnel URL in the Configuration cell (cell 1), e.g.")
    print("  COLAB_JOB_DISPATCHER_URL = 'https://xxx.trycloudflare.com/api'")
else:
    from colab.worker import run_worker_main

    summary = run_worker_main(
        COLAB_JOB_DISPATCHER_URL,
        caps=CAPS,
        max_iterations=MAX_ITERATIONS,
        poll_interval_s=POLL_INTERVAL_S,
    )
    import json
    print(json.dumps(summary, indent=2))

## 7 · Expose Colab's Ollama to the laptop (optional)

In [ ]:
# === Tunnel Ollama (port 11434) so the laptop's ModelRouter can call it ===
# Output prints a https://<...>.trycloudflare.com URL.
# Copy it into the laptop .env as PG_COLAB_OLLAMA_URL and restart the backend.
import subprocess, re

URL = None
try:
    subprocess.run(["which", "cloudflared"], check=True, capture_output=True)
except subprocess.CalledProcessError:
    print("Installing cloudflared...")
    subprocess.run(["curl", "-L", "--fail", "-o", "/usr/local/bin/cloudflared",
                    "https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64"],
                   check=True)
    subprocess.run(["chmod", "+x", "/usr/local/bin/cloudflared"], check=True)

log_path = "/tmp/tunnel.log"
proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--no-autoupdate", "--url", "http://127.0.0.1:11434", "--logfile", log_path],
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL,
)
print("cloudflared started, waiting for tunnel URL...")
import time
for _ in range(20):
    time.sleep(2)
    try:
        content = open(log_path).read()
    except FileNotFoundError:
        continue
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", content)
    if m:
        URL = m.group(0)
        break

if URL:
    print(f"OLLAMA TUNNEL URL: {URL}")
    print("Laptop .env:  PG_COLAB_OLLAMA_URL=" + URL)
else:
    print("Tunnel URL not detected yet \u2014 check the right side log file /tmp/tunnel.log")